Entraîne un Random Forest pour prédire la maladie à partir des symptômes.

Reste volontairement simple : un seul modèle, pas de recherche
d'hyperparamètres, pas de pipeline complexe. L'objectif est d'avoir
une baseline solide et rapide à comprendre/modifier.

Ce notebook part du dataset déjà nettoyé par clean_data.ipynb
(data/cleaned_dataset.csv) : les maladies trop rares, les NaN, etc.
ont déjà été traités là-bas, pas besoin de refaire ce travail ici.

Sorties :
- Le modèle entraîné (models/random_forest_model.pkl)
- La liste des colonnes de symptômes, dans l'ordre attendu par le modèle
  (models/symptom_columns.pkl) -> indispensable pour reconstruire un vecteur
  d'entrée correct côté API, l'ordre des colonnes doit être identique
- Un résumé des performances affiché dans la console

In [1]:
import os
import time

import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, top_k_accuracy_score

In [ ]:
# ---------------------------------------------------------------------------
# qui suppose qu'on lance le notebook depuis ml/notebook.
# ---------------------------------------------------------------------------
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()

# On charge directement le dataset déjà nettoyé par clean_data.ipynb, pas le
# fichier brut -> plus besoin de refiltrer les maladies rares ici.
DATA_PATH = os.path.join(SCRIPT_DIR, "..", "..", "data", "cleaned_dataset.csv")
MODEL_DIR = os.path.join(SCRIPT_DIR, "..", "..", "models")

TARGET_COLUMN = "diseases"
TEST_SIZE = 0.2
RANDOM_STATE = 42

# Pas besoin de forêt gigantesque pour une baseline. À augmenter
# (n_estimators, max_depth) une fois que le pipeline de bout en bout marche.
N_ESTIMATORS = 200

# max_depth=None laisse les arbres pousser sans limite. Avec ~750 maladies à
# distinguer, ça peut produire des arbres énormes -> MemoryError sur une
# machine avec peu de RAM, surtout en parallèle (voir N_JOBS). 
# 30 est un boncompromis pour démarrer
MAX_DEPTH = 30

# min_samples_leaf empêche l'arbre de créer des feuilles avec 1 seul
# exemple, ce qui gonfle inutilement sa taille en mémoire sans vraiment
# améliorer la généralisation.
MIN_SAMPLES_LEAF = 2

# n_jobs=-1 construit plusieurs arbres EN MÊME TEMPS sur plusieurs coeurs,
# donc la mémoire utilisée est multipliée par ce nombre. 
# Sur un laptop avec peu de RAM, mieux vaut limiter à 2-4 plutôt que tout donner d'un coup.
N_JOBS = 2

In [3]:
if not os.path.isfile(DATA_PATH):
    raise FileNotFoundError(
        f"Fichier introuvable : {os.path.abspath(DATA_PATH)}\n"
        f"Avez-vous bien lancé clean_data.ipynb avant, pour générer ce fichier ?"
    )
os.makedirs(MODEL_DIR, exist_ok=True)

print("Chargement des données...")
df = pd.read_csv(DATA_PATH)

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]
symptom_columns = X.columns.tolist()

print(f"{len(df)} lignes, {len(symptom_columns)} symptômes, {y.nunique()} maladies")

Chargement des données...
246823 lignes, 328 symptômes, 721 maladies


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

In [5]:
print(f"Entraînement du Random Forest ({N_ESTIMATORS} arbres)...")
start = time.time()
model = RandomForestClassifier(
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    min_samples_leaf=MIN_SAMPLES_LEAF,
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
)
model.fit(X_train, y_train)
print(f"Entraînement terminé en {time.time() - start:.1f}s")

Entraînement du Random Forest (200 arbres)...
Entraînement terminé en 44.8s


In [6]:
# ------------------------------------------------------------------
# Évaluation : accuracy classique + top-4 accuracy, qui compte une
# prédiction comme correcte si la vraie maladie fait partie des 4
# meilleures probabilités -> c'est cette métrique qui reflète
# vraiment votre mécanique de jeu (4 choix proposés au joueur)
# ------------------------------------------------------------------
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

acc = accuracy_score(y_test, y_pred)
top4_acc = top_k_accuracy_score(y_test, y_proba, k=4, labels=model.classes_)

print("\n--- Résultats ---")
print(f"Accuracy (top-1)  : {acc:.3f}")
print(f"Accuracy (top-4)  : {top4_acc:.3f}  <- pertinent pour vos 4 choix affichés au joueur")


--- Résultats ---
Accuracy (top-1)  : 0.773
Accuracy (top-4)  : 0.890  <- pertinent pour vos 4 choix affichés au joueur


In [7]:
model_path = os.path.join(MODEL_DIR, "random_forest_model.pkl")
columns_path = os.path.join(MODEL_DIR, "symptom_columns.pkl")

joblib.dump(model, model_path)
joblib.dump(symptom_columns, columns_path)

print(f"\nModèle sauvegardé : {os.path.abspath(model_path)}")
print(f"Colonnes sauvegardées : {os.path.abspath(columns_path)}")


Modèle sauvegardé : c:\ESGI\4annee\Trimestre_2\0_PA\MediGuess\models\random_forest_model.pkl
Colonnes sauvegardées : c:\ESGI\4annee\Trimestre_2\0_PA\MediGuess\models\symptom_columns.pkl
